# Convert DDSP TF.js Instrument Models to ONNX

Converts the pretrained DDSP instrument models (violin, flute, trumpet, saxophone)
from TF.js GraphModel format to ONNX for browser inference via ONNX Runtime Web.

**Must run on Google Colab** (x86_64 Linux) — the models were built with TF 2.4
which has no arm64 macOS wheels.

**Run all cells.** Downloads 4 ONNX files at the end.

In [ ]:
# 1. Install a TF version compatible with the old model format
# The models use _FusedConv2D without TArgs attr — need TF < 2.12
!pip install -q "tensorflow==2.11.*" "tf2onnx" "onnx==1.16.*" "numpy<1.24" "protobuf<4"

# Verify
import tensorflow as tf
print(f'TF: {tf.__version__}')

In [ ]:
# 2. Download all 4 instrument models
import os, urllib.request

INSTRUMENTS = ['violin', 'flute', 'trumpet', 'tenor_saxophone']
BASE = 'https://storage.googleapis.com/magentadata/js/checkpoints/ddsp'

for inst in INSTRUMENTS:
    d = f'/tmp/ddsp/{inst}'
    os.makedirs(d, exist_ok=True)
    for f in ['model.json', 'group1-shard1of1.bin']:
        path = f'{d}/{f}'
        if not os.path.exists(path):
            print(f'Downloading {inst}/{f}...')
            urllib.request.urlretrieve(f'{BASE}/{inst}/{f}', path)
        else:
            print(f'{inst}/{f} exists')

print('All models downloaded.')

In [ ]:
# 3. Convert each model: TF.js → ONNX
import subprocess

results = {}
for inst in INSTRUMENTS:
    src = f'/tmp/ddsp/{inst}/model.json'
    dst = f'/tmp/ddsp/ddsp_{inst}.onnx'
    print(f'\nConverting {inst}...')
    r = subprocess.run(
        ['python', '-m', 'tf2onnx.convert', '--tfjs', src, '--output', dst, '--opset', '17'],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        size = os.path.getsize(dst)
        print(f'  ✓ {inst}: {size / 1024 / 1024:.1f} MB')
        results[inst] = dst
    else:
        print(f'  ✗ {inst} failed:')
        # Show last 5 lines of stderr
        for line in r.stderr.strip().split('\n')[-5:]:
            print(f'    {line}')

print(f'\nConverted: {len(results)}/{len(INSTRUMENTS)}')

In [ ]:
# 4. If direct conversion failed, try tfjs → SavedModel → ONNX
if len(results) < len(INSTRUMENTS):
    print('Trying two-step conversion for failed models...')
    !pip install -q tfjs-graph-converter
    
    for inst in INSTRUMENTS:
        if inst in results:
            continue
        src = f'/tmp/ddsp/{inst}/model.json'
        saved = f'/tmp/ddsp/{inst}_saved'
        dst = f'/tmp/ddsp/ddsp_{inst}.onnx'
        
        print(f'\n{inst}: TF.js → SavedModel...')
        r1 = subprocess.run(
            ['tfjs_graph_converter', src, saved, '--output_format', 'tf_saved_model', '--compat_mode'],
            capture_output=True, text=True
        )
        if r1.returncode != 0:
            print(f'  ✗ Step 1 failed: {r1.stderr[-200:]}')
            continue
        
        print(f'{inst}: SavedModel → ONNX...')
        r2 = subprocess.run(
            ['python', '-m', 'tf2onnx.convert', '--saved-model', saved, '--output', dst, '--opset', '17'],
            capture_output=True, text=True
        )
        if r2.returncode == 0:
            size = os.path.getsize(dst)
            print(f'  ✓ {inst}: {size / 1024 / 1024:.1f} MB')
            results[inst] = dst
        else:
            print(f'  ✗ Step 2 failed: {r2.stderr[-200:]}')
    
    print(f'\nConverted: {len(results)}/{len(INSTRUMENTS)}')
else:
    print('All models converted successfully, skipping fallback.')

In [ ]:
# 5. Validate with ONNX Runtime
!pip install -q onnxruntime
import onnxruntime as ort
import numpy as np

for inst, path in results.items():
    sess = ort.InferenceSession(path)
    inputs = {i.name: i for i in sess.get_inputs()}
    print(f'\n{inst}:')
    print(f'  Inputs: {[(i.name, i.shape) for i in sess.get_inputs()]}')
    print(f'  Outputs: {[(o.name, o.shape) for o in sess.get_outputs()]}')
    
    # Build feed with 1 second of A4 at -30 dB
    feed = {}
    for inp in sess.get_inputs():
        shape = [d if isinstance(d, int) else 250 for d in inp.shape]
        if 'f0' in inp.name.lower() or 'pitch' in inp.name.lower():
            feed[inp.name] = np.full(shape, 440.0, dtype=np.float32)
        elif 'loud' in inp.name.lower():
            feed[inp.name] = np.full(shape, -30.0, dtype=np.float32)
        else:
            feed[inp.name] = np.zeros(shape, dtype=np.float32)
    
    out = sess.run(None, feed)
    for i, o in enumerate(sess.get_outputs()):
        print(f'  {o.name}: shape={out[i].shape}, range=[{out[i].min():.4f}, {out[i].max():.4f}]')
    
    ok = any(np.abs(r).max() > 1e-6 for r in out)
    print(f'  {"✓" if ok else "✗"} {"Non-zero output" if ok else "ALL ZEROS"}')

In [ ]:
# 6. Download all ONNX files
from google.colab import files

for inst, path in results.items():
    size = os.path.getsize(path)
    print(f'Downloading ddsp_{inst}.onnx ({size / 1024 / 1024:.1f} MB)...')
    files.download(path)

print(f'\n=== Done ===')
print(f'Upload all 4 files:')
for inst in results:
    print(f'  hf upload jcosta33/vocoder-models ddsp_{inst}.onnx ddsp/{inst}.onnx --repo-type model')